# 形態素解析のユーザ辞書作成と単体テスト

書籍 第5章「言語資源のつくり方」（5-3 辞書作成：5-3-1 仕様、5-3-3 品質管理）に対応する演習ノートブックです。
CPU のみ・十数秒で実行でき、ドメイン固有の専門用語や新語を `SudachiPy` のユーザ辞書へ登録し、解析エラーを修正・検証するプロセスを体験します。

1. **システム辞書のみでの未知語・誤分割エラーの確認**
2. **ユーザ辞書 CSV の作成と `sudachipy ubuild` によるバイナリ辞書ビルド**（5-3-1, 5-3-2）
3. **正規化形（表記揺れ統一）と単体テスト（回帰テスト）による品質管理**（5-3-3）

In [1]:
!pip install -q sudachipy sudachidict_core

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.11/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
import json
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import pandas as pd
import sudachidict_core
import sudachipy
from sudachipy import dictionary, tokenizer

---

## 1. システム辞書のみでの誤分割・表記揺れの確認

書籍の中で登場する専門用語やライブラリ名を含む文を、`SudachiPy` のデフォルト辞書（`sudachidict_core`）だけで解析してみます。

In [3]:
sys_tokenizer = dictionary.Dictionary().tokenizer()

test_sentences = [
    "自然言語処理の教科書はガゼッティアやディスタンススーパービジョンを扱う。",
    "形態素解析にスダチパイやフガシを使う。",
    "翻訳モデルの品質推定にコメットキューイーを用いる。",
]

def show_tokens(tok, sentences):
    for s in sentences:
        items = [f"{m.surface()}({m.part_of_speech()[0]}-{m.part_of_speech()[1]}|{m.normalized_form()})"
                 for m in tok.tokenize(s, tokenizer.Tokenizer.SplitMode.C)]
        print("入力:", s)
        print("  -> ", " / ".join(items))

print("=== システム辞書のみ（登録前） ===")
show_tokens(sys_tokenizer, test_sentences)

=== システム辞書のみ（登録前） ===
入力: 自然言語処理の教科書はガゼッティアやディスタンススーパービジョンを扱う。
  ->  自然言語処理(名詞-普通名詞|自然言語処理) / の(助詞-格助詞|の) / 教科書(名詞-普通名詞|教科書) / は(助詞-係助詞|は) / ガゼッティア(名詞-普通名詞|ガゼッティア) / や(助詞-副助詞|や) / ディスタンス(名詞-普通名詞|ディスタンス) / スーパー(名詞-普通名詞|スーパー) / ビジョン(名詞-普通名詞|ビジョン) / を(助詞-格助詞|を) / 扱う(動詞-一般|扱う) / 。(補助記号-句点|。)
入力: 形態素解析にスダチパイやフガシを使う。
  ->  形態素(名詞-普通名詞|形態素) / 解析(名詞-普通名詞|解析) / に(助詞-格助詞|に) / スダチパイ(名詞-普通名詞|スダチパイ) / や(助詞-副助詞|や) / フガシ(名詞-普通名詞|フガシ) / を(助詞-格助詞|を) / 使う(動詞-一般|使う) / 。(補助記号-句点|。)
入力: 翻訳モデルの品質推定にコメットキューイーを用いる。
  ->  翻訳(名詞-普通名詞|翻訳) / モデル(名詞-普通名詞|モデル) / の(助詞-格助詞|の) / 品質(名詞-普通名詞|品質) / 推定(名詞-普通名詞|推定) / に(助詞-格助詞|に) / コメットキューイー(名詞-普通名詞|コメットキューイー) / を(助詞-格助詞|を) / 用いる(動詞-一般|用いる) / 。(補助記号-句点|。)


`ディスタンススーパービジョン` が `ディスタンス / スーパー / ビジョン` に 3 分割されたり、ツール名・モデル名（`スダチパイ`・`フガシ`・`コメットキューイー`）が `名詞-普通名詞` 扱いで正規化形（`SudachiPy` / `fugashi` / `COMET-QE`）に変換されなかったりしていることがわかります。

---

## 2. ユーザ辞書 CSV の作成とコンパイル（5-3-1, 5-3-2）

書籍 5-3-1 のとおり、**表層形**・**品詞**・**読み**・**正規化形**などを整理したユーザ辞書 CSV を作成し、`sudachipy ubuild` でバイナリ辞書（`user.dic`）をビルドします。
あわせて書籍 5-3-3 に従い、**「なぜそのエントリを追加したか（どのエラーがきっかけだったか）」**のメモも管理テーブルに残しておきます。

In [4]:
# 名詞の標準的な連接コスト ID（4785）を流用して登録
user_entries = [
    # (表層形, 品詞大, 品詞中, 読み, 正規化形, 登録理由メモ)
    ("ディスタンススーパービジョン", "名詞", "普通名詞", "ディスタンススーパービジョン", "ディスタンススーパービジョン", "「ディスタンス/スーパー/ビジョン」に3分割される誤りを修正"),
    ("スダチパイ", "名詞", "固有名詞", "スダチパイ", "SudachiPy", "ライブラリ名のカタカナ表記を SudachiPy に正規化"),
    ("フガシ", "名詞", "固有名詞", "フガシ", "fugashi", "ライブラリ名のカタカナ表記を fugashi に正規化"),
    ("コメットキューイー", "名詞", "固有名詞", "コメットキューイー", "COMET-QE", "固有名詞として登録し正規化形を COMET-QE に統一"),
]

display(pd.DataFrame(user_entries, columns=["表層形", "品詞1", "品詞2", "読み", "正規化形", "エラー記録メモ"]))

# ユーザ辞書用 CSV（18列形式）を書き出し、sudachipy ubuild で user.dic を作成
workdir = Path(tempfile.mkdtemp())
csv_path = workdir / "user_dict.csv"
dic_path = workdir / "user.dic"
cfg_path = workdir / "sudachi.json"

csv_lines = []
for surf, pos1, pos2, reading, norm, _ in user_entries:
    # surface,left_id,right_id,cost,headword,pos1,pos2,pos3,pos4,conj1,conj2,reading,normalized,dic_id,split_type,a_unit,b_unit,word_structure
    row = [surf, "4785", "4785", "4000", surf, pos1, pos2, "一般", "*", "*", "*", reading, norm, "*", "A", "*", "*", "*"]
    csv_lines.append(",".join(row))

csv_path.write_text("\n".join(csv_lines) + "\n", encoding="utf-8")

sys_dic = Path(sudachidict_core.__file__).parent / "resources" / "system.dic"
cand = Path(sys.executable).parent / "sudachipy"
sudachipy_bin = str(cand) if cand.exists() else (shutil.which("sudachipy") or "sudachipy")
subprocess.run(
    [sudachipy_bin, "ubuild", "-s", str(sys_dic), "-o", str(dic_path), str(csv_path)],
    check=True,
)

# ユーザ辞書を有効にする設定ファイルを作成
base_cfg = json.loads((Path(sudachipy.__file__).parent / "resources" / "sudachi.json").read_text(encoding="utf-8"))
base_cfg["systemDict"] = str(sys_dic)
base_cfg["userDict"] = [str(dic_path)]
for key in ("characterDefinitionFile",):
    if key in base_cfg and not Path(base_cfg[key]).is_absolute():
        base_cfg[key] = str(Path(sudachipy.__file__).parent / "resources" / base_cfg[key])
cfg_path.write_text(json.dumps(base_cfg, ensure_ascii=False, indent=2), encoding="utf-8")

user_tokenizer = dictionary.Dictionary(config_path=str(cfg_path)).tokenizer()
print("=== ユーザ辞書適用後 ===")
show_tokens(user_tokenizer, test_sentences)

,表層形,品詞1,品詞2,読み,正規化形,エラー記録メモ
0,ディスタンススーパービジョン,名詞,普通名詞,ディスタンススーパービジョン,ディスタンススーパービジョン,「ディスタンス/スーパー/ビジョン」に3分割される誤りを修正
1,スダチパイ,名詞,固有名詞,スダチパイ,SudachiPy,ライブラリ名のカタカナ表記を SudachiPy に正規化
2,フガシ,名詞,固有名詞,フガシ,fugashi,ライブラリ名のカタカナ表記を fugashi に正規化
3,コメットキューイー,名詞,固有名詞,コメットキューイー,COMET-QE,固有名詞として登録し正規化形を COMET-QE に統一


/tmp/tmppk614c7t/user_dict.csv -> 4 in 0.00 sec
resolve -> 3 in 2.90 sec
pos_table -> 2 in 0.00 sec
word_id table -> 8 in 0.00 sec
trie -> 1024 in 0.00 sec
strings -> 110 in 0.00 sec
entries -> 256 in 0.00 sec
reference_id_table -> 1 in 0.00 sec
=== ユーザ辞書適用後 ===
入力: 自然言語処理の教科書はガゼッティアやディスタンススーパービジョンを扱う。
  ->  自然言語処理(名詞-普通名詞|自然言語処理) / の(助詞-格助詞|の) / 教科書(名詞-普通名詞|教科書) / は(助詞-係助詞|は) / ガゼッティア(名詞-普通名詞|ガゼッティア) / や(助詞-副助詞|や) / ディスタンススーパービジョン(名詞-普通名詞|ディスタンススーパービジョン) / を(助詞-格助詞|を) / 扱う(動詞-一般|扱う) / 。(補助記号-句点|。)
入力: 形態素解析にスダチパイやフガシを使う。
  ->  形態素(名詞-普通名詞|形態素) / 解析(名詞-普通名詞|解析) / に(助詞-格助詞|に) / スダチパイ(名詞-固有名詞|SudachiPy) / や(助詞-副助詞|や) / フガシ(名詞-固有名詞|fugashi) / を(助詞-格助詞|を) / 使う(動詞-一般|使う) / 。(補助記号-句点|。)
入力: 翻訳モデルの品質推定にコメットキューイーを用いる。
  ->  翻訳(名詞-普通名詞|翻訳) / モデル(名詞-普通名詞|モデル) / の(助詞-格助詞|の) / 品質(名詞-普通名詞|品質) / 推定(名詞-普通名詞|推定) / に(助詞-格助詞|に) / コメットキューイー(名詞-固有名詞|COMET-QE) / を(助詞-格助詞|を) / 用いる(動詞-一般|用いる) / 。(補助記号-句点|。)


---

## 3. 単体テストによる品質管理（5-3-3）

書籍 5-3-3 では、辞書に新しいエントリを追加したら**実際にその単語を含む文を解析して意図通りに動くか確認し、エラー記録や単体テストとして残しておく**重要性が述べられています。
ここでは登録した4語について、期待する表層形で分割され、正規化形が一致するかをテストします。既存の分割への影響も確認するには、登録語を含まない文を回帰テストに加え、辞書の追加前後で解析結果を比較します。

In [5]:
def run_dictionary_unit_tests(tok):
    cases = [
        ("ディスタンススーパービジョンを学ぶ", "ディスタンススーパービジョン", "ディスタンススーパービジョン"),
        ("形態素解析にスダチパイを使う", "スダチパイ", "SudachiPy"),
        ("ラッパーのフガシを試す", "フガシ", "fugashi"),
        ("コメットキューイーで評価する", "コメットキューイー", "COMET-QE"),
    ]
    for sent, expected_surface, expected_norm in cases:
        morphemes = list(tok.tokenize(sent, tokenizer.Tokenizer.SplitMode.C))
        matched = [m for m in morphemes if m.surface() == expected_surface]
        assert matched, f"分割失敗: {sent!r} に {expected_surface!r} が見つかりません -> {[m.surface() for m in morphemes]}"
        assert matched[0].normalized_form() == expected_norm, (
            f"正規化不一致: expected={expected_norm}, actual={matched[0].normalized_form()}"
        )
    print(f"全 {len(cases)} 件の辞書単体テストに合格しました！")

run_dictionary_unit_tests(user_tokenizer)

全 4 件の辞書単体テストに合格しました！
